In [ ]:
import re
import emoji
import pandas as pd
import kagglehub
from nltk.lm import MLE
from nltk.tokenize import word_tokenize
from nltk.lm.preprocessing import padded_everygram_pipeline
from sklearn.model_selection import train_test_split
from nltk.util import ngrams
from nltk.lm.preprocessing import pad_both_ends

In [2]:
path = kagglehub.dataset_download("adizafar/large-random-tweets-from-pakistan")
df = pd.read_csv(f"{path}/Random Tweets from Pakistan- Cleaned- Anonymous.csv", encoding_errors="ignore", low_memory=False)
tweets = df["full_text"].dropna()
print(len(tweets))
tweets.head()

202151


0    تیرا لیڈر میرا لیڈر نواز شریف نواز شریف❤️  کیا...
1    Happy birthday to my brother n boss , May you ...
2                                                 ❤️❤️
3    `suspicious °jikook au jimin'in yaşadığı kasab...
4    Speaking of @SpiderMan... 😂   https://t.co/uGJ...
Name: full_text, dtype: str

In [3]:
def clean(text):
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"\bRT\b", "", text)
    text = re.sub(r"http\S+|www\.\S+", "", text) 
    text = re.sub(r"@\w+", "", text)
    text = emoji.replace_emoji(text, "")
    return text.lower().strip()

tweets = tweets.apply(clean)
tweets = tweets[tweets != ""]

print(len(tweets))
tweets.head()

187673


0    تیرا لیڈر میرا لیڈر نواز شریف نواز شریف  کیا آ...
1    happy birthday to my brother n boss , may you ...
3    `suspicious °jikook au jimin'in yaşadığı kasab...
4                                      speaking of ...
5                                  alexa, skip to 2021
Name: full_text, dtype: str

In [4]:
tokenized = [word_tokenize(t) for t in tweets]
train_sents, test_sents = train_test_split(tokenized, test_size=0.2, random_state=42)
train, vocab = padded_everygram_pipeline(2, train_sents)
lm = MLE(2)
lm.fit(train, vocab)
print(len(lm.vocab))

142143


In [5]:
def bigrams_of(sents):
    return [bg for s in sents for bg in ngrams(pad_both_ends(s, n=2), 2)]

print("Train perplexity:", lm.perplexity(bigrams_of(train_sents[:100])))
print("Test perplexity:", lm.perplexity(bigrams_of(test_sents[:100])))

Train perplexity: 56.25654586033164
Test perplexity: inf


In [6]:
print("Perplexity of a bigram seen in training:", lm.perplexity([("pakistan", "is")]))
print()
print("P(is | pakistan) =", lm.score("is", ["pakistan"]))
print()
print("P(pakistan) =", lm.score("pakistan"))
print("Perplexity of 'pakistan' =", lm.perplexity([("pakistan",)]))

Perplexity of a bigram seen in training: 19.914700544464612

P(is | pakistan) = 0.05021416203408366

P(pakistan) = 0.0036001506597268702
Perplexity of 'pakistan' = 277.76615328533694


In [7]:
for i in range(5):
    print(" ".join(w for w in lm.generate(15, text_seed=["pakistan"], random_seed=i) if w != "</s>"))

started registering cases . due to solve students get ready to launch e pehn ly
. they saved karachi has found out that : . then i stop !
zinda rahengi us . then there is evident from pakistan have any hour after that
20 people did no one 's take an award پاکستان وہ پھر رمضان میں
..... : is an afghan peace remains the saath nahi hota ha administration
